# Accessing a CIRA Forecast

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/cira_forecast.ipynb)

This notebook demonstrates how to access a CIRA MLWP forecast from the icechunk store and evaluate it against GHCNh station observations. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench import inputs, metrics, cases, evaluate
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2021 Pacific NW Heat Dome — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9001,
    title="2021 Pacific NW Heat Dome (demo)",
    start_date=datetime.datetime(2021, 6, 27),
    end_date=datetime.datetime(2021, 6, 30),
    location=BoundingBoxRegion.create_region(
        latitude_min=46.0,
        latitude_max=52.0,
        longitude_min=236.0,
        longitude_max=242.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
fcnv2 = inputs.get_cira_icechunk(model_name="FOUR_v200_IFS")

In [ ]:
ghcn_target = inputs.GHCN()

metrics_list = [
    metrics.MeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
    metrics.MaximumMeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
    metrics.CriticalSuccessIndex(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
        forecast_threshold=310,
        target_threshold=310,
    ),
]

In [ ]:
evaluation_object = [
    inputs.EvaluationObject(
        event_type="heat_wave",
        metric_list=metrics_list,
        target=ghcn_target,
        forecast=fcnv2,
    ),
]

ewb_runner = evaluate.ExtremeWeatherBench(
    case_metadata=cases,
    evaluation_objects=evaluation_object,
)

output = ewb_runner.run_evaluation()
print(output)